# Monte Carlo simulations

In [1]:
import numpy as np
import pandas as pd
import sys
import plotly.express as px
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.utils import load_config
from src.monte_carlo import run_simulation_grid
from src.metrics import summarize_df
from src.figures import rmse_frontier, bias_dml, bias_ols, sd_dml, sd_ols, coverage_dml, coverage_ols

c:\Users\danil\anaconda3\envs\vcausal\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Running MC Sim

In [2]:
config = load_config("baseline")

df = run_simulation_grid(config, save_each=True)

alpha_y=1, alpha_d=1: 100%|██████████| 500/500 [12:34<00:00,  1.51s/it]


In [3]:
results = summarize_df(df)
results.head()

,estimator,bias,sd,rmse,coverage,avg_ci_length,alpha_y,alpha_d
0,OLS,-0.009151,0.092213,0.092574,0.946,0.357421,0.0,0.00
1,DML,-0.008345,0.111066,0.111268,0.932,0.417350,0.0,0.00
2,OLS,0.003513,0.094821,0.094791,0.942,0.358503,0.0,0.25
3,DML,-0.016995,0.111758,0.112933,0.948,0.419674,0.0,0.25
4,OLS,0.003706,0.093857,0.093836,0.942,0.361143,0.0,0.50


## Frontiers


In [4]:
rmse_frontier(results)

In [5]:
bias_ols(results)

In [6]:
bias_dml(results)

In [7]:
sd_ols(results)

In [8]:
sd_dml(results)

In [14]:
rmse_wide = results.pivot_table(
    index=["alpha_y", "alpha_d"],
    columns="estimator",
    values="rmse"
    ).reset_index()

rmse_wide["rmse_diff"] = rmse_wide["OLS"] - rmse_wide["DML"]


rmse_wide.head()


estimator,alpha_y,alpha_d,DML,OLS,rmse_diff
0,0.0,0.00,0.111268,0.092574,-0.018694
1,0.0,0.25,0.112933,0.094791,-0.018142
2,0.0,0.50,0.117646,0.093836,-0.023810
3,0.0,0.75,0.139848,0.093398,-0.046450
4,0.0,1.00,0.170001,0.093135,-0.076866


In [16]:

fig = px.density_heatmap(
    rmse_wide,
    x="alpha_d",
    y="alpha_y",
    z="rmse_diff",
    histfunc="avg",
    color_continuous_scale="RdBu",
    title="RMSE Difference: OLS - DML"
    )

fig.update_layout(
    xaxis_title="alpha_d",
    yaxis_title="alpha_y",
    coloraxis_colorbar_title="RMSE diff"
    )


fig.show()

In [21]:
import numpy as np
import plotly.graph_objects as go

# pivot should have alpha_y as index and alpha_d as columns
# values = rmse_diff = OLS RMSE - DML RMSE
pivot = rmse_wide.pivot(index="alpha_y", columns="alpha_d", values="rmse_diff")

z = pivot.values
x_vals = list(pivot.columns)
y_vals = list(pivot.index)

text = np.round(z, 3)

fig = go.Figure(data=go.Heatmap(
    z=z,
    x=x_vals,
    y=y_vals,
    colorscale="RdBu",
    zmid=0,
    text=text,
    texttemplate="%{text}",
    textfont={"size": 12},
    colorbar=dict(title="OLS RMSE - DML RMSE"),
    hovertemplate="alpha_d=%{x}<br>alpha_y=%{y}<br>RMSE diff=%{z:.4f}<extra></extra>"
))

fig.update_layout(
    title={"text": "Misspecification Frontier: RMSE Difference (OLS - DML)", "x": 0.5},
    xaxis=dict(
        title="Treatment assignment complexity (alpha_d)",
        tickmode="array",
        tickvals=x_vals,
        ticktext=[str(v) for v in x_vals]
    ),
    yaxis=dict(
        title="Outcome complexity (alpha_y)",
        tickmode="array",
        tickvals=y_vals,
        ticktext=[str(v) for v in y_vals]
    )
)


fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()